In [1]:
from XGBoostModel import XGBoostModel
from RunData import RunPipeline
import pandas as pd
import optuna
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold ,train_test_split, cross_val_score
import kagglehub
import os
import zipfile
from seed_utils import SEED
import requests


/home/daniel7/.conda/envs/my_env_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# GenericDataPipeline

In [2]:
class GenericDataPipeline:

    def rank_features(self, X, y, n_folds=5):
        print(f"Starting feature importance calculation with XGBoost and {n_folds}-fold CV...")

        X_copy = X.copy()

        null_ratios = {}
        for col in X_copy.columns:
            null_ratios[col] = X_copy[col].isna().mean()

        for col in X_copy.select_dtypes(include=['int64', 'float64']).columns:
            X_copy[col] = X_copy[col].replace([np.inf, -np.inf], np.nan)
            
            non_null = X_copy[col].dropna()
            if len(non_null) > 0:
                mean_val = non_null.mean()
                std_val = non_null.std()
                if std_val > 0 and not np.isnan(mean_val):
                    upper_bound = mean_val + 5 * std_val
                    lower_bound = mean_val - 5 * std_val
                    X_copy[col] = X_copy[col].clip(lower=lower_bound, upper=upper_bound)
        
        num_cols = X_copy.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = X_copy.select_dtypes(include=['object']).columns
        
        X_processed = X_copy.copy()
        
        for col in cat_cols:
            if X_processed[col].isna().sum() > 0:
                most_freq = X_processed[col].mode()[0] if not X_processed[col].isna().all() else 'Unknown'
                X_processed[col] = X_processed[col].fillna(most_freq)
            X_processed[col] = X_processed[col].astype('category')
        
        for col in num_cols:
            if X_processed[col].isna().sum() > 0:
                median_val = X_processed[col].median() if not X_processed[col].isna().all() else 0
                X_processed[col] = X_processed[col].fillna(median_val)
        
        X_filled = X_processed

        mean_target = float(np.mean(y))
        print(f"Mean target value (for base_score): {mean_target:.6f}")

        base_score = 0.5
        if 0 < mean_target < 1:
            base_score = mean_target

        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': 6,
            'eta': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'alpha': 0.01,
            'lambda': 1.0,
            'gamma': 0.1,
            'base_score': base_score, 
            'seed': 42,
            'tree_method': 'hist',
            'device': 'cuda'

        }

        kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

        fold_importances = []

        fold_scores = []

        print(f"Training XGBoost models across {n_folds} folds...")

        y_values = set(y.unique())
        print(f"Unique target values: {y_values}")

        if not all(isinstance(val, (int, float)) for val in y_values):
            print("Warning: Target variable contains non-numeric values. Converting to numeric.")
            y = y.astype(float)

        if not all(val in [0, 1] for val in y_values):
            print("Warning: Target variable contains values other than 0 and 1.")
            print("Converting target to binary: 0 for negative class, 1 for positive class.")
            y = (y > 0).astype(int)

        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled, y)):
            print(f"Fold {fold+1}/{n_folds}")

            X_train, X_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            train_pos = np.mean(y_train)
            val_pos = np.mean(y_val)
            print(f"  Train positive rate: {train_pos:.4f}, Validation positive rate: {val_pos:.4f}")

            try:
                dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
                dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

                # Train model
                model = xgb.train(
                    params,
                    dtrain,
                    num_boost_round=100,
                    early_stopping_rounds=20,
                    evals=[(dval, 'validation')],
                    verbose_eval=False
                )

                y_pred = model.predict(dval)
                score = roc_auc_score(y_val, y_pred)
                fold_scores.append(score)

                try:
                    importance = model.get_score(importance_type='gain')

                    fold_importance = {col: importance.get(col, 0) for col in X_filled.columns}

                    max_value = max(fold_importance.values()) if max(fold_importance.values()) > 0 else 1
                    normalized_importance = {col: val/max_value for col, val in fold_importance.items()}

                    fold_importances.append(normalized_importance)
                except Exception as e:
                    print(f"Warning: Could not get feature importance for fold {fold+1}: {str(e)}")

            except Exception as e:
                print(f"Error in fold {fold+1}: {str(e)}")
                print("Skipping this fold and continuing with others.")

        if fold_scores:
            mean_auc = np.mean(fold_scores)
            print(f"Mean AUC across {len(fold_scores)} successful folds: {mean_auc:.4f}")
        else:
            print("No successful folds. Cannot calculate mean AUC.")

        if not fold_importances:
            print("No feature importance information could be gathered from any fold.")
            print("Using a simple fallback method for feature scoring.")

            avg_importance = {}
            for col in X_filled.columns:
                try:
                    valid_mask = ~pd.isna(X_copy[col]) & ~pd.isna(y)
                    if valid_mask.sum() > 10: 
                        if X_filled[col].dtype.name == 'category':
                            col_values = X_filled[col][valid_mask].cat.codes
                        else:
                            col_values = X_filled[col][valid_mask]
                        
                        y_values = y[valid_mask]
                        corr = abs(np.corrcoef(col_values, y_values)[0, 1])
                        avg_importance[col] = corr if not np.isnan(corr) else 0
                    else:
                        avg_importance[col] = np.random.uniform(0, 0.001)
                except Exception as e:
                    print(f"Warning: Could not calculate correlation for {col}: {str(e)}")
                    avg_importance[col] = np.random.uniform(0, 0.001)
        else:
            avg_importance = {}
            for col in X_filled.columns:
                importances = [fold_imp.get(col, 0) for fold_imp in fold_importances]
                if importances:
                    avg_importance[col] = np.mean(importances)
                else:
                    null_ratio = null_ratios.get(col, 0)
                    avg_importance[col] = max(0.001 * (1 - null_ratio), 0.0001)  

        feature_data = []
        for col in X_copy.columns:
            gain_score = avg_importance.get(col, 0)
            null_ratio = null_ratios.get(col, 0)
            feature_data.append({
                'feature_name': col,
                'gain_score': gain_score,
                'null_ratio': null_ratio
            })
        
        feature_df = pd.DataFrame(feature_data)
        feature_df = feature_df.sort_values('gain_score', ascending=False)
        
        print("\nFeature scores:")
        print("-" * 100)
        print(f"{'Rank':<5} | {'Feature':<30} | {'Gain Score':<15} | {'Null Ratio':<10}")
        print("-" * 100)

        for i, row in feature_df.iterrows():
            rank = i + 1
            feat = row['feature_name']
            gain = row['gain_score']
            null_ratio = row['null_ratio']
            print(f"{rank:<5} | {feat:<30} | {gain:.6f} | {null_ratio:.4f}")

        return feature_df

    def objective(self, trial, feature_scores_with_nulls, all_features_score, df, label="readmitted"):
        K = 10
        selected_features = feature_scores_with_nulls['feature_name'].to_list()[:K]
        print("selected_features")
        print(selected_features)
        all_features = all_features_score['feature_name'].to_list()
        # Create binary assignment for each of the selected features:
        assignment = {}
        for feat in selected_features:
            # 1 means feature goes to stage1, 0 means feature is used in stage2
            assignment[feat] = trial.suggest_categorical(f"assign_{feat}", [1, 0])
        
        for feat in all_features:
            if feat not in assignment.keys():
                assignment[feat] = 1
        vals = 0
        for key,value in assignment.items():
            vals +=value
        if vals==0:
            return 99999
        penalty = 0 
        # Derive feature sets based on the assignment:
        stage1_features = [feat for feat, assign in assignment.items() if assign == 1]
        stage2_features = [feat for feat, assign in assignment.items() if assign == 0]
        csv_name = f"trial_{trial.number}_results.csv"
        dm = RunPipeline()
        validation_loss = dm.full_run(df,stage1_features,stage2_features,label,csv_name)
        if validation_loss == 999:
            return validation_loss
        else:
            final_objective = validation_loss + penalty
            return final_objective
    def preprocessing(self,df):
        for col in df.columns:
            if df[col].dtype == 'object':
                # Replace ? with NaN
                df[col] = df[col].replace(['?', ''], np.nan)
                # Convert to categorical codes
                if df[col].isna().sum() < len(df):  # If not all values are NaN
                    df[col] = pd.Categorical(df[col]).codes

                    # Ensure -1 (missing) values are converted to NaN
                    df[col] = df[col].replace(-1, np.nan)
        # Convert boolean columns to int
        for col in df.select_dtypes(include=['bool']).columns:
            df[col] = df[col].astype(int)
        return df 

    def Airlines(self,n_trials=20):
        path = kagglehub.dataset_download("manishkumar7432698/airline-passangers-booking-data")
        csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
        
        csv_path1 = os.path.join(path, csv_files[1]) 
        data1 = pd.read_csv(csv_path1)
        csv_path3 = os.path.join(path, csv_files[3])  
        data3 = pd.read_csv(csv_path3)
        df = pd.merge(
            data1,
            data3,
            on=['flight_number'],
            how='left'
        )
        df = df[df['satisfaction_type'].notna()]

        df['satisfaction_type'] = df['satisfaction_type'].map({
            'Dissatisfied': 0,
            'Satisfied': 1
        })
        
        columns_to_drop  = ['flight_number','origin_station_code_x','destination_station_code_x','verbatim_text','transformed_text',
                            'origin_station_code_y','destination_station_code_y','record_locator','score','arrival_delay_minutes','generation',
                            'equipment_type_code','actual_flown_miles','departure_gate','arrival_gate']
        df = df.drop(columns=columns_to_drop)
        df = self.preprocessing(df)
        return df


# [Airline Customer Holiday Booking Dataset](https://www.kaggle.com/datasets/manishkumar7432698/airline-passangers-booking-data/data?select=Survey+data_Inflight+Satisfaction+Score.csv)

In [6]:

fs = GenericDataPipeline()
study = fs.Airlines(20)

Using 'Customer Status' as target variable
Starting feature importance calculation with XGBoost and 5-fold CV...
Mean target value (for base_score): 0.358725
Training XGBoost models across 5 folds...
Unique target values: {np.int64(0), np.int64(1)}
Fold 1/5
  Train positive rate: 0.3587, Validation positive rate: 0.3587
Fold 2/5
  Train positive rate: 0.3587, Validation positive rate: 0.3587
Fold 3/5
  Train positive rate: 0.3587, Validation positive rate: 0.3587
Fold 4/5
  Train positive rate: 0.3587, Validation positive rate: 0.3587
Fold 5/5
  Train positive rate: 0.3587, Validation positive rate: 0.3587


[I 2025-05-18 13:55:01,700] A new study created in memory with name: no-name-045b8e1b-dc3a-4669-ada7-c913c74e1124
[I 2025-05-18 13:55:01,823] Trial 0 finished with value: 999.0 and parameters: {'assign_cabin_name': 1, 'assign_loyalty_program_level_y': 1, 'assign_loyalty_program_level_x': 1}. Best is trial 0 with value: 999.0.


Mean AUC across 5 successful folds: 0.6882

Feature scores:
----------------------------------------------------------------------------------------------------
Rank  | Feature                        | Gain Score      | Null Ratio
----------------------------------------------------------------------------------------------------
28    | international_domestic_indicator | 0.985713 | 0.0000
18    | cabin_code_desc                | 0.792748 | 0.0000
31    | hub_spoke                      | 0.782922 | 0.0000
20    | entity_y                       | 0.740222 | 0.0000
5     | entity_x                       | 0.721932 | 0.0000
27    | haul_type                      | 0.699783 | 0.0000
19    | cabin_name                     | 0.698718 | 0.3835
23    | loyalty_program_level_y        | 0.640453 | 0.2479
17    | arrival_delay_group_y          | 0.624316 | 0.0000
13    | scheduled_departure_date_y     | 0.614390 | 0.0000
24    | fleet_type_description_y       | 0.605466 | 0.0000
22    | seat_fact

[I 2025-05-18 13:55:01,936] Trial 1 finished with value: 999.0 and parameters: {'assign_cabin_name': 1, 'assign_loyalty_program_level_y': 1, 'assign_loyalty_program_level_x': 1}. Best is trial 0 with value: 999.0.
[I 2025-05-18 13:55:02,042] Trial 2 finished with value: 999.0 and parameters: {'assign_cabin_name': 1, 'assign_loyalty_program_level_y': 1, 'assign_loyalty_program_level_x': 1}. Best is trial 0 with value: 999.0.



=== Breakdown BEFORE splitting ===
has_extended
0    202813
Name: count, dtype: int64
Extended percentage: 0.0 %
❌ Not enough variation in has_extended — all rows have or all rows lack extended features!
selected_features
['cabin_name', 'loyalty_program_level_y', 'loyalty_program_level_x']

=== Breakdown BEFORE splitting ===
has_extended
0    202813
Name: count, dtype: int64
Extended percentage: 0.0 %
❌ Not enough variation in has_extended — all rows have or all rows lack extended features!
selected_features
['cabin_name', 'loyalty_program_level_y', 'loyalty_program_level_x']

=== Breakdown BEFORE splitting ===
has_extended
1    185473
0     17340
Name: count, dtype: int64
Extended percentage: 91.45 %


...

********** run results **********
{'max_depth': 5, 'learning_rate': 0.0020587810884466744, 'n_estimators': 243, 'min_child_weight': 1, 'subsample': 0.7499172010415533, 'colsample_bytree': 0.8215691855466589, 'gamma': 2.708102472327583, 'lambda': 4.567250864947747, 'alpha': 5.761132187110672, 'scale_pos_weight': 2.3493394403105308, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.6022348730661555), np.float64(0.5932081817351902), np.float64(0.6001426647548713)]
********** run results **********


/sise/home/daniel7/IncrementalLearing/Changes /XGBoostModel.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
[I 2025-05-18 15:11:10,325] Trial 19 finished with value: 0.5867962076910241 and parameters: {'max_depth': 3, 'learning_rate': 0.010584610857850766, 'n_estimators': 372, 'min_child_weight': 2, 'subsample': 0.9701053310819874, 'colsample_bytree': 0.7429048599216728, 'gamma': 1.8145680729635547, 'lambda': 9.884995704560668, 'alpha': 0.1105696922609618, 'scale_pos_weight': 3.856305607154261}. Best is trial 11 with value: 0.553327580824364.


********** run results **********
{'max_depth': 3, 'learning_rate': 0.010584610857850766, 'n_estimators': 372, 'min_child_weight': 2, 'subsample': 0.9701053310819874, 'colsample_bytree': 0.7429048599216728, 'gamma': 1.8145680729635547, 'lambda': 9.884995704560668, 'alpha': 0.1105696922609618, 'scale_pos_weight': 3.856305607154261, 'random_state': 42, 'eval_metric': 'auc', 'tree_method': 'hist', 'device': 'cuda'}
*****
[np.float64(0.5890655857662805), np.float64(0.5817244599879474), np.float64(0.5895985773188446)]
********** run results **********
Best parameters found: {'max_depth': 3, 'learning_rate': 0.003656365656370924, 'n_estimators': 110, 'min_child_weight': 1, 'subsample': 0.9866611078652573, 'colsample_bytree': 0.7848332984342834, 'gamma': 1.615557365707534, 'lambda': 9.947456909838646, 'alpha': 3.719577245600835, 'scale_pos_weight': 0.17048931852521088}
Best CV AUC score: 0.5533

===== Detailed Cross-Validation Results with Best Parameters =====
Params: {'max_depth': 3, 'learn

[I 2025-05-18 15:11:14,863] Trial 19 finished with value: -0.00746590563011118 and parameters: {'assign_cabin_name': 0, 'assign_loyalty_program_level_y': 0, 'assign_loyalty_program_level_x': 0}. Best is trial 16 with value: -0.023086126942419072.


Test set AUC (with extended features): 0.5519
Test set AUC (without extended features): 0.5520
Overall test set AUC: 0.5522
international_domestic_indicator: 0.0000
cabin_code_desc: 0.0588
hub_spoke: 0.0555
entity_y: 0.0000
entity_x: 0.0000
haul_type: 0.1132
arrival_delay_group_y: 0.0988
scheduled_departure_date_y: 0.0458
fleet_type_description_y: 0.1312
seat_factor_band_y: 0.0467
fleet_type_description_x: 0.1533
response_group_y: 0.0186
number_of_legs: 0.0656
media_provider: 0.0382
fleet_usage_x: 0.0000
arrival_delay_group_x: 0.0000
seat_factor_band_x: 0.0000
response_group_x: 0.0000
scheduled_departure_date_x: 0.0000
departure_delay_group: 0.0000
Unnamed: 0: 0.0000
sentiments: 0.0000
fleet_usage_y: 0.0000
ua_uax: 0.0000
driver_sub_group1: 0.0000
question_text: 0.0000
ques_verbatim_text: 0.0000
driver_sub_group2: 0.0000
cabin_name: 0.1164
loyalty_program_level_y: 0.0382
loyalty_program_level_x: 0.0000
has_extended: 0.0198

Top 10 features by importance:
fleet_type_description_x: 0.153

# Final Results

AUC Summary:
                         Model      AUC
      Base model (no extended) 0.554443
Extended model (with extended) 0.569891
  Combined model (no extended) 0.542510
Combined model (with extended) 0.558738

Base Features:
international_domestic_indicator, cabin_code_desc, hub_spoke, entity_y, entity_x, haul_type, arrival_delay_group_y, scheduled_departure_date_y, fleet_type_description_y, seat_factor_band_y, fleet_type_description_x, response_group_y, number_of_legs, media_provider, fleet_usage_x, arrival_delay_group_x, seat_factor_band_x, response_group_x, scheduled_departure_date_x, departure_delay_group, Unnamed: 0, sentiments, fleet_usage_y, ua_uax, driver_sub_group1, question_text, ques_verbatim_text, driver_sub_group2

Extended Features:
cabin_name, loyalty_program_level_y, loyalty_program_level_x, has_extended

AUC Differences:
Extended model - Combined (with extended): 0.01115300000000008
Base model - Combined (no extended):       0.011932999999999971

Standard Deviations:
Extended model - Std Dev: 0.0029
Base model - Std Dev:     0.0013
